# M2 Gap — LSTM SHAP Interpretability

**Issue:** #49
**Owner:** Nelson
**Reviewer:** Mitchel
**Branch:** `artifact/m2-lstm-shap`

## Objective

Wire SHAP (SHapley Additive exPlanations) values to the trained LSTM's output, per
proposal §5.3 and §8, so the model's predictions are interpretable rather than a
black box.

## Scope

Explains the LSTM's 1-day-ahead output (the operationally primary horizon) using
`shap.GradientExplainer` against the final full-sample model trained in
`notebooks/03_models/lstm_baseline.ipynb`. Reports SHAP values **per feature, per
observation** (not just a single global importance ranking), satisfying #49's
definition of done.

Each explained observation gets one SHAP value per feature per day in its 20-day
lookback window; these are summed across the lookback window to get one
per-feature-per-observation attribution, matching the granularity a reviewer would
need to answer "which macro variable drove this specific forecast."


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import shap

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from lstm_baseline import (  # noqa: E402
    load_common_sample,
    make_windows,
    load_final_model,
    FEATURES,
)

MODEL_PATH = PROJECT_ROOT / "outputs" / "lstm_final_model.pt"
N_EXPLAIN = 250   # most recent forecast origins to explain
N_BACKGROUND = 100  # background sample for the expected-value baseline

print("Model path:", MODEL_PATH)
print("Features:", FEATURES)


Model path: /home/nholguin/projects/DAMO-699-Capstone-project-GRP5/outputs/lstm_final_model.pt
Features: ['d_yield_spread_10y_2y', 'd_overnight_rate', 'd_us_treasury_10y', 'd_fed_funds_rate', 'd_cpi_yoy', 'd_usdcad']


/home/nholguin/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = load_common_sample()
X, Y, origin_idx = make_windows(df)
model, scaling = load_final_model(MODEL_PATH, n_features=len(FEATURES))

X_scaled = (X - scaling["x_mean"]) / scaling["x_std"]
X_t = torch.from_numpy(X_scaled.astype(np.float32))

print("Windowed observations:", X_t.shape)
print("Explaining the most recent", N_EXPLAIN, "forecast origins")


Windowed observations: torch.Size([4228, 20, 6])
Explaining the most recent 250 forecast origins


In [3]:
class Horizon1Wrapper(nn.Module):
    """SHAP needs a single scalar output per observation; this exposes only the
    1-day-ahead head (index 0) of the 3-horizon LSTM."""

    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, x):
        return self.base_model(x)[:, 0:1]


wrapped_model = Horizon1Wrapper(model)
wrapped_model.eval()

background = X_t[-(N_EXPLAIN + N_BACKGROUND):-N_EXPLAIN]
explain_set = X_t[-N_EXPLAIN:]
explain_origins = origin_idx[-N_EXPLAIN:]
explain_dates = df.loc[explain_origins, "date"].reset_index(drop=True)

explainer = shap.GradientExplainer(wrapped_model, background)
shap_values = explainer.shap_values(explain_set)

shap_values = np.asarray(shap_values)  # (n_explain, lookback, n_features, 1)
print("SHAP values shape:", shap_values.shape)


SHAP values shape: (250, 20, 6, 1)


In [4]:
# Sum each feature's SHAP contribution across the lookback window -> one
# per-feature-per-observation value, not just a single global ranking.
per_feature_per_obs = shap_values[..., 0].sum(axis=1)  # (n_explain, n_features)

shap_df = pd.DataFrame(per_feature_per_obs, columns=FEATURES)
shap_df.insert(0, "origin_date", explain_dates)

print("Per-feature, per-observation SHAP values:", shap_df.shape)
shap_df.head()


Per-feature, per-observation SHAP values: (250, 7)


,origin_date,d_yield_spread_10y_2y,d_overnight_rate,d_us_treasury_10y,d_fed_funds_rate,d_cpi_yoy,d_usdcad
0,2025-06-18,0.001195,-0.003971,-0.012462,0.0,0.003960,0.009116
1,2025-06-19,0.000502,-0.004276,0.005530,0.0,0.006345,0.014467
2,2025-06-20,0.001678,-0.001896,0.011911,0.0,0.005036,0.016509
3,2025-06-23,0.000210,-0.004171,0.014793,0.0,0.004683,0.009944
4,2025-06-24,-0.000070,-0.004449,-0.001226,0.0,0.004573,0.005947


In [5]:
# Global summary (mean |SHAP|) -- supplementary to, not a replacement for, the
# per-observation table above.
summary = (
    shap_df[FEATURES]
    .abs()
    .mean()
    .sort_values(ascending=False)
    .rename("mean_abs_shap")
    .reset_index()
    .rename(columns={"index": "feature"})
)
summary


,feature,mean_abs_shap
0,d_us_treasury_10y,0.014764
1,d_usdcad,0.008063
2,d_cpi_yoy,0.006185
3,d_yield_spread_10y_2y,0.005851
4,d_overnight_rate,0.004974
5,d_fed_funds_rate,0.003137


In [6]:
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

shap_df.to_csv(OUTPUT_DIR / "r3_lstm_shap_values.csv", index=False)
summary.to_csv(OUTPUT_DIR / "r3_lstm_shap_summary.csv", index=False)

print("Saved SHAP outputs:")
print("- r3_lstm_shap_values.csv  (per-feature, per-observation)")
print("- r3_lstm_shap_summary.csv (per-feature mean |SHAP| ranking)")


Saved SHAP outputs:
- r3_lstm_shap_values.csv  (per-feature, per-observation)
- r3_lstm_shap_summary.csv (per-feature mean |SHAP| ranking)


In [7]:
print("LSTM SHAP Conclusion")
print("=" * 60)
print(f"Explained {len(shap_df)} of the most recent 1-day-ahead forecasts")
print(f"Background sample size: {N_BACKGROUND}")
print()
print("Feature importance ranking (mean |SHAP|, 1-day horizon):")
for _, row in summary.iterrows():
    print(f"  {row['feature']:<25} {row['mean_abs_shap']:.6f}")


LSTM SHAP Conclusion
Explained 250 of the most recent 1-day-ahead forecasts
Background sample size: 100

Feature importance ranking (mean |SHAP|, 1-day horizon):
  d_us_treasury_10y         0.014764
  d_usdcad                  0.008063
  d_cpi_yoy                 0.006185
  d_yield_spread_10y_2y     0.005851
  d_overnight_rate          0.004974
  d_fed_funds_rate          0.003137
